In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
conn = sqlite3.connect('../data/olist.db')

orders = pd.read_sql('SELECT * FROM orders', conn)
order_items = pd.read_sql('SELECT * from order_items', conn)
reviews = pd.read_sql('SELECT * FROM reviews', conn)
customers = pd.read_sql('SELECT * FROM customers', conn)

conn.close()

In [3]:
## filtr dostarczonych

delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'{len(delivered)} dostarczonych z {len(orders)} zamowionych')

96478 dostarczonych z 99441 zamowionych


In [4]:
## merge z order_items po order_id zeby miec ceny do KPI

df = delivered.merge(order_items, on='order_id', how='inner')

In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,110197,96478,8272b63d03f5f79c56e9e4120aec44ef,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,110197,96478,fc3d1daec319d62d49bfb5e1f83123e9,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,110197,1,delivered,110197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,110197,95956,2017-07-16 18:19:25,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_approved_at,110182,88274,2018-02-24 03:20:27,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_carrier_date,110195,80106,2018-05-09 15:48:00,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_customer_date,110189,95658,2017-07-31 18:03:02,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_estimated_delivery_date,110197,445,2017-12-20 00:00:00,596,NaN,NaN,NaN,NaN,NaN,NaN,NaN
delivery_days,110189.0,NaN,NaN,NaN,12.007342,9.451153,0.0,6.0,10.0,15.0,209.0
order_item_id,110197.0,NaN,NaN,NaN,1.198181,0.706676,1.0,1.0,1.0,1.0,21.0


In [6]:
## KPI Sprzedażowe              
    ## Revenue, Orders, Average Order Value (AOV), Mediana wartości zamówienia

    # Revenue: suma cen produktów (tylko delivered) — bez kosztów frachtu                                                                                                                                      
    # AOV: średnia zawyżona przez drogie zamówienia, mediana (87 BRL) lepiej oddaje typowy zakup 

total_revenue = df['price'].sum()
total_orders = df['order_id'].nunique()
aov = total_revenue / total_orders
median_order_value = df.groupby('order_id')['price'].sum().median()

print(f'Revenue: {total_revenue:,.2f} BRL')
print(f'Orders: {total_orders:,}')
print(f'AOV: {aov:.2f} BRL')
print(f'Median Order Value: {median_order_value:.2f} BRL')

Revenue: 13,221,498.11 BRL
Orders: 96,478
AOV: 137.04 BRL
Median Order Value: 86.57 BRL


In [7]:
## KPI logistyczne
    ## Sredni czas dostawy, Mediana czasu dostawy, % dostaw w czasie

    # Delivery: średnia 12.1 vs mediana 10  punkty oddalone od centrum logistycznego ciagna srednia do gory                                                                                                                              
    # On-Time: 92% na czas to jest solidny wynik, ale 8% spóźnień to ok. 7.7k zamówień
    # dostawy powyzej 100 dni: 63 z 96k — to 0.065%, marginalny ułamek.

avg_delivery = delivered['delivery_days'].mean()
median_delivery = delivered['delivery_days'].median()

    ##konwersja delivered i estimated na daty

delivered['order_delivered_customer_date'] = pd.to_datetime(delivered['order_delivered_customer_date'])
delivered['order_estimated_delivery_date'] = pd.to_datetime(delivered['order_estimated_delivery_date'])

delivered['is_late'] = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']
on_time_pct = (1 - delivered['is_late'].mean()) * 100

delivery_over_100 = len(delivered[delivered['delivery_days'] > 100][['order_id', 'delivery_days', 'order_status']])



print(f'Sredni czas dostawy: {avg_delivery:.2f} dni')
print(f'Mediana czasu dostawy: {median_delivery} dni')
print(f'Procent dostaw w czasie: {on_time_pct:.2f}%')
print(f'Czas dostawy powyzej 100 dni: {delivery_over_100}')

Sredni czas dostawy: 12.09 dni
Mediana czasu dostawy: 10.0 dni
Procent dostaw w czasie: 91.89%
Czas dostawy powyzej 100 dni: 63


In [8]:
## KPI satysfakcji
    ## srednia ocena satysfakcji, % z 5 gwiazdkami, % z 1+2 gwiazdkami

    # Rating 4.09 — dobra średnia, ale 15% krytyków (1-2★) to sygnał do poprawy                                                                                                                                
    # Korelacja z delivery: im dłuższa dostawa, tym gorszy rating (potwierdzone w SQL) 

avg_reviews =reviews['review_score'].mean()
pct_5star = (reviews['review_score'] == 5).mean() * 100
pct_1_2star = (reviews['review_score'] <= 2).mean() * 100

print(f'Srednia ocena: {avg_reviews:.2f}')
print(f'% 5 gwiazdek: {pct_5star:.2f}')
print(f'% 1 i 2 gwiazdki: {pct_1_2star:.2f}')

Srednia ocena: 4.09
% 5 gwiazdek: 57.78
% 1 i 2 gwiazdki: 14.69


In [9]:
## polaczenie tabel delivered i customer
    # do obliczenia KPI retencji potrzebny jest customer_unique_id

cust_orders = delivered.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='inner')

cust_orders.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,96478,96478,e481f51cbdc54678b7cc49136f2d6af7,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,96478,96478,9ef432eb6251297304e76186b10a928d,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,96478,1,delivered,96478,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,96478,95956,2018-03-31 15:08:21,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_approved_at,96464,88274,2018-02-27 04:31:10,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_carrier_date,96476,80106,2018-05-09 15:48:00,47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_customer_date,96470,NaN,NaN,NaN,2018-01-14 12:41:33.581683,2016-10-11 13:46:32,2017-09-25 22:15:09.500000,2018-02-02 19:32:21,2018-05-15 22:54:48.500000,2018-10-17 13:22:46,NaN
order_estimated_delivery_date,96478,NaN,NaN,NaN,2018-01-25 17:09:52.325711,2016-10-04 00:00:00,2017-10-05 00:00:00,2018-02-16 00:00:00,2018-05-28 00:00:00,2018-10-25 00:00:00,NaN
delivery_days,96470.0,NaN,NaN,NaN,12.093604,0.0,6.0,10.0,15.0,209.0,9.55138
is_late,96478,2,False,88652,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
## KPI liczba zamowien przez klienta

orders_per_customer = cust_orders.groupby('customer_unique_id')['order_id'].nunique()
orders_per_customer.sort_values(ascending=False).head(15)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    15
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
dc813062e0fc23409cd255f7f53c7074     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
f0e310a6839dce9de1638e0fe5ab282a     6
63cfc61cee11cbe306bff5857d00bfe4     6
b4e4f24de1e8725b74e4a1f4975116ed     5
5e8f38a9a1c023f3db718edcf926a2db     5
4e65032f1f574189fb793bac5a867bbc     5
35ecdf6858edc6427223b64804cf028e     5
74cb1ad7e6d5674325c1f99b5ea30d82     5
Name: order_id, dtype: int64

In [11]:
## KPI retrencji
    # liczba klientow, liczba powracajacych klientow, srednie zamowienie per klient

    # Tylko 3% klientów wraca, nad tym trzeba popracowac,  największe wyzwanie biznesowe Olist                                                                                                                                            
    # Avg 1.03 zamówienia/klient, platforma działa jako marketplace jednorazowego zakupu.

total_customers = len(orders_per_customer)
returning = (orders_per_customer > 1).sum()
returning_pct = returning / total_customers * 100
avg_orders = orders_per_customer.mean()

print(f'Unikalni klienci: {total_customers:,}')                                                                                                                                                            
print(f'Powracający: {returning:,} ({returning_pct:.1f}%)')                                                                                                                                                
print(f'Avg zamówień/klient: {avg_orders:.2f}') 

Unikalni klienci: 93,358
Powracający: 2,801 (3.0%)
Avg zamówień/klient: 1.03


In [12]:
## KPI tabela podsumowujaca

kpi_summary = pd.DataFrame([
    ['Total Revenue', f'{total_revenue:,.0f} BRL'],
    ['Total Orders', f'{total_orders:,}'],                                                                                                                                                                 
    ['AOV (średnia)', f'{aov:.2f} BRL'],                                                                                                                                                                   
    ['AOV (mediana)', f'{median_order_value:.2f} BRL'],                                                                                                                                                    
    ['Avg Delivery Days', f'{avg_delivery:.1f}'],                                                                                                                                                          
    ['Median Delivery Days', f'{median_delivery:.0f}'],
    ['On-Time Delivery %', f'{on_time_pct:.1f}%'],                                                                                                                                                         
    ['Avg Review Score', f'{avg_reviews:.2f}'],
    ['% 5-gwiazdkowych', f'{pct_5star:.1f}%'],                                                                                                                                                             
    ['% 1-2 gwiazdkowych', f'{pct_1_2star:.1f}%'],
    ['Returning Customers %', f'{returning_pct:.1f}%'],                                                                                                                                                    
    ['Avg Orders/Customer', f'{avg_orders:.2f}'],
], columns=['KPI', 'Wartość'])

kpi_summary 

,KPI,Wartość
0,Total Revenue,"13,221,498 BRL"
1,Total Orders,"96,478"
2,AOV (średnia),137.04 BRL
3,AOV (mediana),86.57 BRL
4,Avg Delivery Days,12.1
5,Median Delivery Days,10
6,On-Time Delivery %,91.9%
7,Avg Review Score,4.09
8,% 5-gwiazdkowych,57.8%
9,% 1-2 gwiazdkowych,14.7%
